In [1]:
import sys
import numpy as np

# sys.path.append("../../../src/")
from Rain.Rain import Rain
# sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-07 20:57:21.241716: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-07 20:57:21.918819: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {
        "num_of_workers": 3,
        "ips": ['127.0.0.1', '127.0.0.1', '127.0.0.1'], #[,'127.0.0.1', '127.0.0.1', '127.0.0.1'], 
        "ports": [50151, 50152, 50153]
        
      }
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "iterations": 3,
  "chunk_size": 9 * 1024*1024,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 5,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-07 20:57:22,952 [DEBUG] [Rain] Rain is initialized
2023-07-07 20:57:22,953 [DEBUG] [Provisioner] Creating coordinator
2023-07-07 20:57:22,954 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-07 20:57:22,956 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-07 20:57:22,957 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-07 20:57:22,958 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-07 20:57:22,959 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-07 20:57:22,960 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [8]:
# model = rain.train(X_train, y_train, strategy='async')

In [9]:
# X_test, y_test = get_test_data()
# loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
# print("\nTest accuracy: %.1f%%" % (100.0 * acc))

In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-07 20:57:22,986 [INFO] [Provisioner] provisioner is serving
2023-07-07 20:57:22,988 [DEBUG] [Provisioner] Starting coordinator
2023-07-07 20:57:22,991 [INFO] [Coordinator] coordinator is serving
2023-07-07 20:57:22,992 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-07 20:57:23,000 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-07 20:57:23,002 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-07 20:57:23,004 [DEBUG] [Coordinator] coordinator is sending the number of workers to provisioner
2023-07-07 20:57:23,010 [DEBUG] [Provisioner] Provision requested the coordinator to get the number of workers
2023-07-07 20:57:23,011 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-07 20:57:23,012 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker_50151/
2023-07-07 20:57:23,015 [INFO] [Worker_50151] Worker is running 

Epoch 1/5
Epoch 1/5
Epoch 1/5
157/157 [==============================] - 3s 8ms/step - loss: 0.7140 - accuracy: 0.7721
Epoch 2/5
157/157 [==============================] - 3s 8ms/step - loss: 0.7000 - accuracy: 0.7776
Epoch 2/5
157/157 [==============================] - 1s 8ms/step - loss: 0.2965 - accuracy: 0.9124
Epoch 3/5
157/157 [==============================] - 1s 8ms/step - loss: 0.2337 - accuracy: 0.9304
Epoch 4/5
157/157 [==============================] - 1s 8ms/step - loss: 0.2310 - accuracy: 0.9302
Epoch 4/5
157/157 [==============================] - 1s 8ms/step - loss: 0.1908 - accuracy: 0.9427
Epoch 5/5
157/157 [==============================] - 1s 8ms/step - loss: 0.2028 - accuracy: 0.9379
Epoch 5/5
Epoch 5/5
157/157 [==============================] - 1s 7ms/step - loss: 0.1657 - accuracy: 0.9509



2023-07-07 20:57:33,285 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
2023-07-07 20:57:33,286 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
2023-07-07 20:57:33,295 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
2023-07-07 20:57:33,296 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_1_trained.pkl from worker2


sending data to divider


2023-07-07 20:57:33,339 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-07 20:57:33,348 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_1_trained.pkl from worker2 successfully
2023-07-07 20:57:37,282 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
2023-07-07 20:57:37,283 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_1_trained.pkl from worker3
2023-07-07 20:57:37,337 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_1_trained.pkl from worker3 successfully
2023-07-07 20:57:37,359 [DEBUG] [DeepLearning] Iteration 1/3 complete.
2023-07-07 20:57:37,359 [DEBUG] [DeepLearning] Starting iteration 2/3
2023-07-07 20:57:37,379 [DEBUG] [DividerAmbassador] 127.0.0.1:50151
2023-07-07 20:57:37,379 [DEBUG] [DividerAmbassador] 127.0.0.1:50152
2023-07-07 20:57:37,379 [DEBUG] [DividerAmbassador] 127.0.0.1:50153
2023-07

...

sending data to divider
157/157 [==============================] - 1s 5ms/step - loss: 0.0809 - accuracy: 0.9747


2023-07-07 20:57:56,851 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker2


sending data to divider


2023-07-07 20:57:56,852 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_3_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_3_trained.pkl from worker2
2023-07-07 20:57:56,881 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
2023-07-07 20:57:56,902 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
2023-07-07 20:57:57,338 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker3
2023-07-07 20:57:57,340 [DEBUG] [DividerAmbassador] divider begins downloading .

sending data to divider


In [11]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.0738 - accuracy: 0.9786

Test accuracy: 97.9%
